# E1 · Baseline · Chủ: Người 1 · Phục vụ Chương 5 (mục 5.1)
1. Dò `regParam` trên **validation** → **T5.1**
2. Huấn luyện mô hình chính thức `lr_base`, `lr_def`; chọn ngưỡng vận hành trên validation → lưu `thresholds.json` (Người 2 và E2 dùng)
3. Đo **một lần** trên test, lặp 3 seed → **T5.2** (trung bình ± độ lệch chuẩn)
4. Phân tích theo LLM / lĩnh vực / cách giải mã, phân bố điểm → **F5.1, F5.2, T5.3, F5.3**

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "ten-cua-ban"     # điền tên để log kết quả
REPO  = "https://github.com/<ten-tai-khoan>/ai-text-detection.git"   # sửa thành repo của nhóm

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

In [ ]:
from aitext import data, models, evaluate
train, val, test = data.load_splits(spark)
train = data.with_class_weight(train).cache(); val = val.cache(); test = test.cache()
print(train.count(), val.count(), test.count())

## 1. Dò regParam trên validation → T5.1 (khoảng 15 phút)

In [ ]:
sweep_tr = train.sample(0.3, seed=config.SEED).cache()
rows = []
for rp in config.REG_PARAMS:
    t0 = time.time()
    m = models.fit(sweep_tr, defended=True, reg_param=rp)
    p = evaluate.score(m, val, models.text_col(True)).cache()
    thr = evaluate.thr_at_fpr(p)
    rows.append({"regParam": rp, "AUROC (val)": evaluate.auroc(p), "TPR@FPR5% (val)": evaluate.rate_above(p, 1, thr),
                 "FPR@0.5 (val)": evaluate.rate_above(p, 0, 0.5), "ngưỡng FPR5%": thr, "giây": time.time() - t0})
    p.unpersist()
t51 = pd.DataFrame(rows); display(t51.round(4))
BEST_RP = float(t51.sort_values(["TPR@FPR5% (val)", "AUROC (val)"], ascending=False).iloc[0]["regParam"])
models.save_thresholds(reg_param=BEST_RP)
evaluate.save_table(t51, "T5.1_regparam", notes=f"Dò trên 30% train, đo trên validation. Chọn regParam={BEST_RP}.")
print("==> regParam =", BEST_RP)

## 2. Mô hình chính thức + ngưỡng vận hành (khoảng 15 phút)

In [ ]:
m_base = models.fit(train, defended=False, reg_param=BEST_RP)
m_def  = models.fit(train, defended=True,  reg_param=BEST_RP)
models.save(m_base, "lr_base"); models.save(m_def, "lr_def")
thr_base = evaluate.thr_at_fpr(evaluate.score(m_base, val, models.text_col(False)))
thr_def  = evaluate.thr_at_fpr(evaluate.score(m_def,  val, models.text_col(True)))
print(models.save_thresholds(thr_base=thr_base, thr_def=thr_def, fpr_target=config.FPR_TARGET))

## 3. Kết quả trên test, lặp 3 seed → T5.2 (khoảng 35–40 phút)
Mỗi seed: huấn luyện trên 80% train (lấy mẫu ngẫu nhiên theo seed), chọn ngưỡng trên validation, đo trên test.

In [ ]:
runs = []
for s in config.SEEDS:
    sub = train.sample(0.8, seed=s).cache()
    for name, d in [("baseline", False), ("defended", True)]:
        m = models.fit(sub, defended=d, reg_param=BEST_RP)
        thr = evaluate.thr_at_fpr(evaluate.score(m, val, models.text_col(d)))
        r = evaluate.metrics(evaluate.score(m, test, models.text_col(d)), thr)
        evaluate.log_run("E1", OWNER, f"{name}|seed={s}|rp={BEST_RP}", r)
        runs.append({"mô hình": name, "seed": s, **r})
    sub.unpersist()
runs = pd.DataFrame(runs)
cols = ["auroc", "tpr", "fpr", "fpr_at_0.5", "tpr_at_fpr_oracle"]
t52 = runs.groupby("mô hình")[cols].agg(["mean", "std"]).round(4)
display(t52)
evaluate.save_table(runs, "T5.2_seeds_raw"); evaluate.save_table(t52.reset_index(), "T5.2_clean_mean_std")

## 4. Phân tích sâu (mô hình defended chính thức) → F5.1, F5.2, T5.3, F5.3

In [ ]:
pred = evaluate.score(m_def, test, models.text_col(True)).cache()
by_model  = pd.DataFrame(evaluate.auc_by(pred, "model")).sort_values("auroc")
by_domain = pd.DataFrame(evaluate.auc_by(pred, "domain")).sort_values("auroc")
by_dec    = pd.DataFrame(evaluate.auc_by(pred, "decoding")).sort_values("auroc")
evaluate.save_table(by_model, "F5.1_auc_by_model"); evaluate.save_table(by_domain, "F5.2_auc_by_domain")
evaluate.save_table(by_dec, "T5.3_auc_by_decoding"); display(by_dec)
for df_, key, code_ in [(by_model, "model", "F5.1_auc_by_model"), (by_domain, "domain", "F5.2_auc_by_domain")]:
    fig, ax = plt.subplots(figsize=(6, 4)); df_.plot.barh(x=key, y="auroc", ax=ax, legend=False, xlim=(0.5, 1))
    ax.set_title(code_); evaluate.save_figure(fig, code_); plt.show()

In [ ]:
th = models.load_thresholds()
fp_dom = (pred.filter("label = 0").groupBy("domain")
          .agg(F.avg((F.col("score") > 0.5).cast("double")).alias("fpr_0_5"),
               F.avg((F.col("score") > th["thr_def"]).cast("double")).alias("fpr_op")).toPandas())
evaluate.save_table(fp_dom, "T5.3b_fpr_by_domain"); display(fp_dom.round(3))
h = pred.select("label", "score").sample(0.3, seed=config.SEED).toPandas()
fig, ax = plt.subplots(figsize=(7, 3.5))
for lab, name in [(0, "người"), (1, "AI")]:
    ax.hist(h[h.label == lab]["score"], bins=50, alpha=.6, density=True, label=name)
ax.axvline(0.5, color="gray", ls="--", label="0,5"); ax.axvline(th["thr_def"], color="red", ls="--", label="ngưỡng vận hành")
ax.set_yscale("log"); ax.set_title("F5.3 Phân bố điểm P(AI) trên test"); ax.legend()
evaluate.save_figure(fig, "F5.3_score_hist"); plt.show()

## Ghi chú cho Chương 5 (3–5 câu, chép vào `T5.2_notes.md` hoặc gửi Người 2)
- ...